# Single-emitter ESM observation series + intelligence-report generation

This notebook demonstrates generation and saving of synthetic single-aircraft emitter tracks for the experiments described in `docs/single_emitter_observation_series.md`.

In addition to radar/kinematics observations, every observation is enriched with 10--12 synthetic `IntelligenceReport` records. Reports cover operator, aircraft-variant, aircraft-family, radar-type, radar-mode, location, and relation-hypothesis claim types. They include collection/publication timestamps near the ESM observation timestamp and intentionally mix correct and contradictory claims for fusion experiments.

Per-observation ground-truth labels and synthetic report truth markers are retained for evaluation, but downstream inference/classification views strip truth fields to avoid leakage.


In [ ]:
from pathlib import Path
import json
import os
from collections import Counter

from esm_observation_series_generator import (
    generate_observation_series_with_intelligence_reports,
    observations_without_ground_truth,
)
from rgcn_fusion.intelligence_reports import build_report_evidence_rows, flatten_reports_from_series


In [ ]:
# Use multiple worker processes so this demo exercises the parallelized series generator.
# Cap the notebook demo to a few workers to avoid oversubscribing shared environments.
worker_count = min(12, os.cpu_count() or 1)

data = generate_observation_series_with_intelligence_reports(
    count=2500,
    seed=42,
    intelligence_seed=4200,
    min_duration_s=1.0,
    max_duration_s=60.0,
    sample_interval_s=0.5,
    mode_switch_probability=0.08,  # sampled between adjacent observations; entries can switch several times
    workers=worker_count,
)

reports = flatten_reports_from_series(data)
claims = [claim for report in reports for claim in report["claims"]]
len(data["observation_series"]), data["metadata"], len(reports), Counter(c["claim_type"] for c in claims)


In [ ]:
data["observation_series"][0]

In [ ]:
first_series = data["observation_series"][0]
inference_rows = observations_without_ground_truth(first_series)
first_obs = first_series["observations"][0]
first_reports = first_obs["intelligence_reports"]
{
    "series_id": first_series["series_id"],
    "observation_count": first_series["observation_count"],
    "duration_s": first_series["duration_s"],
    "mode_shift_sequence_indices": first_series["mode_shift_sequence_indices"],
    "ground_truth_modes": [truth["mode"] for truth in first_series["ground_truth_mode_sequence"][:10]],
    "reports_for_first_observation": len(first_reports),
    "first_report_time_window": (first_reports[0]["collected_at"], first_reports[0]["published_at"]),
    "first_report_claim": first_reports[0]["claims"][0],
    "inference_row_has_ground_truth": "ground_truth_label" in inference_rows[0],
    "first_observation_keys": list(first_obs.keys()),
    "first_inference_row_keys": list(inference_rows[0].keys()),
}


In [ ]:
all_observations = [obs for series in data["observation_series"] for obs in series["observations"]]
evidence_rows = build_report_evidence_rows(all_observations)
{name: len(rows) for name, rows in evidence_rows.items()}


In [ ]:
output_path = Path("../generated/demo_esm_observation_series_with_intel.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
# Also refresh the legacy demo path so existing notebooks can opt into the enriched payload.
legacy_output_path = Path("../generated/demo_esm_observation_series.json")
legacy_output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
output_path, legacy_output_path
